# doc-extraction — OmniDocBench full benchmark (Kaggle)

Thin notebook: install, configure paths, run, summarize. All extraction and
evaluation logic lives in the repository (`experiments/005_omnidocbench/`,
`src/doc_extraction/evaluation/omnidocbench.py`) — nothing is implemented
in this notebook. See `experiments/005_omnidocbench/README.md` for the full
design, the pinned OmniDocBench commit, and what has and hasn't been
run locally (a small demo subset only — CPU time on the dev machine made
the full 1651-page benchmark impractical there; that's what this notebook
is for).

**Before running**: attach the OmniDocBench dataset as a Kaggle Dataset
input (upload `OmniDocBench.json` + `images/` from
https://huggingface.co/datasets/opendatalab/OmniDocBench, or add it as a
Kaggle Dataset if already published there) and note its mount path under
`/kaggle/input/`.

**No private data**: only the public OmniDocBench dataset is used here.
Do not attach the `doc-extraction` repo's own sample documents (they are
gitignored and not part of this clone) or any other private input.

## 1. Install

In [ ]:
%cd /kaggle/working
!git clone https://github.com/<your-fork-or-org>/doc-extraction.git
%cd doc-extraction

# Main pipeline (this Kaggle image's Python — check it's 3.10+; the
# extraction backends have no upper Python-version pin).
!pip install -e ".[docling,tables]" -q

# The OmniDocBench evaluator, cloned outside the repo per the main project's
# convention (see experiments/005_omnidocbench/README.md "Setup") and pinned
# to the same commit that adapter was verified against.
!git clone https://github.com/opendatalab/OmniDocBench.git .external/OmniDocBench
%cd .external/OmniDocBench
!git checkout 193627ae9e97d89188468ed1ee3b7a856ff76044
%cd /kaggle/working/doc-extraction

# Kaggle images ship Python >=3.10 as the default interpreter, which the
# evaluator's own `>=3.10,<3.12` constraint accepts directly — no separate
# venv needed here the way the Windows dev machine required one. If the
# attached image happens to be 3.12+, create an isolated env instead:
#   !python -m venv .venv-omnidoc && .venv-omnidoc/bin/pip install -e .external/OmniDocBench
# and pass --omnidoc-python .venv-omnidoc/bin/python to the commands below.
!pip install -e .external/OmniDocBench -q

## 2. Configure paths

Edit `DATASET_PATH` to match your attached input.

In [ ]:
import os

DATASET_PATH = "/kaggle/input/omnidocbench"  # <-- edit to your attached dataset's mount path
OUTPUT_ROOT = "/kaggle/working/results"

# Keep model downloads off the small root volume — same reasoning as the
# dev machine's C:\ constraint, see docs/setup.md "Cache redirection".
os.environ.setdefault("HF_HOME", "/kaggle/working/.cache/huggingface")
os.environ.setdefault("DOCLING_ARTIFACTS_PATH", "/kaggle/working/.cache/docling")
os.environ.setdefault("XDG_CACHE_HOME", "/kaggle/working/.cache")

assert os.path.exists(DATASET_PATH), f"DATASET_PATH does not exist: {DATASET_PATH} — check the attached input's mount path"

## 3. Run — baseline backend

Omit `--subset` for the full dataset once the small run below looks right.

In [ ]:
# Validate the integration on a small, deterministic subset first — exactly
# the same principle as the local CPU validation (see the parent README).
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_smoke \
    --subset 20 \
    --match-workers 2

In [ ]:
# Full run. --match-workers: keep to roughly 1/3-1/2 of the instance's CPU
# count (upstream's own guidance, to avoid deadlocks/OOM in its worker pools).
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline \
    --match-workers 4

## 4. Run — Docling backend

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend docling \
    --output {OUTPUT_ROOT}/docling \
    --match-workers 4

## 5. GPU backends (not run on the dev machine — see docs/backends.md)

If this notebook is attached to a GPU instance and a GPU-capable backend
(e.g. MinerU, PaddleOCR-VL) has been added to `doc_extraction`'s backend
registry, install its extra and point `--backend` at it the same way:

```python
!pip install -e ".[mineru]" -q   # once the extra is actually implemented
!python experiments/005_omnidocbench/run.py --dataset {DATASET_PATH} --backend mineru --output {OUTPUT_ROOT}/mineru
```

As of this phase, `mineru`/`paddleocr`/`vlm` are documented-unavailable
stubs (`doc_extraction.pipelines.base.BackendUnavailableError`) — see
`docs/backends.md`. Enabling one is future work, not something this
notebook assumes is already done.

## 6. Summarize

In [ ]:
import json
from pathlib import Path

for backend_dir in sorted(Path(OUTPUT_ROOT).glob("*")):
    report = backend_dir / "report.md"
    if report.exists():
        print(f"===== {backend_dir.name} =====")
        print(report.read_text(encoding="utf-8"))
        print()

In [ ]:
# Download results/ from the Kaggle output panel afterward, or copy the
# small committed-shape files (report.md, metrics.json, runtime.json,
# run_metadata.json) back into the repo's experiments/005_omnidocbench/results/
# tree for the same commit-what's-small convention the local run used — see
# the parent README's "Files" section.